In [1]:
!pip install fsspec

In [2]:
import pandas as pd
import re
from torchvision import datasets as tv_datasets
from google.colab import drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# image dataset
TRAIN_DIR = '/content/drive/MyDrive/INTERNSHIP/PlantVillage/train'

image_dataset = tv_datasets.ImageFolder(TRAIN_DIR)
image_classes_raw = image_dataset.classes

def clean_label(label):
    label = label.split("___")[-1]
    label = label.replace("_", " ").lower()
    label = re.sub(r'[^a-z\s]', '', label)
    return label.strip()

image_classes_clean = [clean_label(cls) for cls in image_classes_raw]

total_image_classes = len(image_classes_clean)

image_disease_classes = [cls for cls in image_classes_clean if "healthy" not in cls]
total_image_diseases = len(image_disease_classes)

print("IMAGE DATASET")
print("Total Classes:", total_image_classes)
print("Total Disease Classes:", total_image_diseases)

IMAGE DATASET
Total Classes: 38
Total Disease Classes: 26


In [5]:
# text dataset
df = pd.read_parquet(
    "hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet"
)

print("Columns in text dataset:", df.columns)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Columns in text dataset: Index(['image', 'caption', 'captions'], dtype='object')


In [6]:
def clean_text(text):
    text = str(text).lower()
    text = text.replace("_", " ")
    text = re.sub(r'[^a-z\s]', '', text)
    return text.strip()

df["clean_caption"] = df["caption"].apply(clean_text)

text_classes_detected = set()

for cls in image_classes_clean:
    for caption in df["clean_caption"]:
        if cls in caption:
            text_classes_detected.add(cls)
            break

text_classes_detected = sorted(text_classes_detected)

total_text_classes = len(text_classes_detected)

text_disease_classes = [cls for cls in text_classes_detected if "healthy" not in cls]
total_text_diseases = len(text_disease_classes)

print("\nTEXT DATASET")
print("Total Classes Detected:", total_text_classes)
print("Total Disease Classes Detected:", total_text_diseases)


TEXT DATASET
Total Classes Detected: 8
Total Disease Classes Detected: 7


In [10]:
image_disease_set = set(image_disease_classes)
text_disease_set = set(text_disease_classes)


matched_diseases = sorted(image_disease_set.intersection(text_disease_set))

print("\nMATCHED DISEASES")
print("------------------")

matched_table = pd.DataFrame({
    "Matched Diseases": matched_diseases
})

display(matched_table)

print("Total Matched Diseases:", len(matched_diseases))


unmatched_diseases = sorted(
    (image_disease_set - text_disease_set) |
    (text_disease_set - image_disease_set)
)

print("\nUNMATCHED DISEASES")
print("--------------------")

unmatched_table = pd.DataFrame({
    "Unmatched Diseases": unmatched_diseases
})

display(unmatched_table)

print("Total Unmatched Diseases:", len(unmatched_diseases))


MATCHED DISEASES
------------------


,Matched Diseases
0,bacterial spot
1,early blight
2,late blight
3,leaf mold
4,septoria leaf spot
5,target spot
6,tomato mosaic virus


Total Matched Diseases: 7

UNMATCHED DISEASES
--------------------


,Unmatched Diseases
0,apple scab
1,black rot
2,cedar apple rust
3,cercospora leaf spot gray leaf spot
4,common rust
5,esca black measles
6,haunglongbing citrus greening
7,leaf blight isariopsis leaf spot
8,leaf scorch
9,northern leaf blight


Total Unmatched Diseases: 13
